# チューリング・トーク ― 判定と発話内容の分析

判定の内訳・自由記述・質問内容を読むためのノートブック。

**このノートブックはリポジトリに入れない**(`.gitignore` 済み)。
`pandas` などの解析用ライブラリを `requirements.txt` に混ぜないための分離でもある。
Render 側の `requirements.txt` は触らないこと。

## 分析の対象

1. 品質検査(4項目のみ)
2. 判定の内訳 ― 真の正体 × 判定ラベルのクロス表
3. **誤警報の分離保存** ― 人間を「AI」と判定した事例を切り出して残す
4. 自由記述の頻出語 ― 判定結果の別に分けて比較
5. 質問内容の分析 ― 何を聞いたか、いつ聞いたか、当たった質問は何か
6. 決め手アノテーション
7. 会話全文ビューア

## 分析の性質についての注意

ここで行うのは**探索的分析**であり、仮説検定ではない。
頻出語や分類別の正答率は「次に何を仮説として立てるか」を決めるための材料で、
これ自体を結論として報告することはできない。
特に、探索で見つけた差をそのまま検定にかけると多重比較の問題が生じる。

In [ ]:
# ---- 設定 ----------------------------------------------------
# データの取り込み方は3通り。どれか1つを使う。
#   (a) Webの /admin/export を保存したJSONが1本だけ → JSON_PATH に指定
#   (b) JSONが複数ある(推奨)  → 先に `python import_exports.py` で
#       merged.db にまとめてから DB_PATH="merged.db"
#       ※ 複数のエクスポートは中身が重複し、messages.id も衝突する。
#          単純に連結してはいけない。import_exports.py がこれを処理する。
#   (c) 手元でアプリを動かした場合 → DB_PATH="../game_data.db"
DB_PATH   = "merged.db"
JSON_PATH = None

# 集計から外すもの(品質検査の結果に応じて切り替える)
EXCLUDE_DUMMY = True           # ダミー応答を含むゲーム
EXCLUDE_V0    = True           # v0(素のモデル)のゲーム
ENTRY_TYPE    = None           # "quick" / "single" / "private_group" / None(全部)

OUT_DIR = "out"                # 書き出し先
# --------------------------------------------------------------

import json, os, re, sqlite3, unicodedata
from collections import Counter, defaultdict

import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
os.makedirs(OUT_DIR, exist_ok=True)

TABLES = ("games", "players", "messages", "annotations", "surveys")

def load():
    if not JSON_PATH and not os.path.isfile(DB_PATH):
        raise FileNotFoundError(
            f"{DB_PATH} が見つかりません。DB_PATH を直すか、"
            "JSON_PATH に /admin/export の出力を指定してください。")
    if JSON_PATH:
        with open(JSON_PATH, encoding="utf-8") as f:
            raw = json.load(f)
        return {t: pd.DataFrame(raw.get(t, [])) for t in TABLES}
    conn = sqlite3.connect(DB_PATH)
    out = {}
    for t in TABLES:
        try:
            out[t] = pd.read_sql_query(f"SELECT * FROM {t}", conn)
        except Exception:
            out[t] = pd.DataFrame()
    conn.close()
    return out

D = load()
games, players, messages = D["games"], D["players"], D["messages"]
annotations, surveys = D["annotations"], D["surveys"]

print({t: len(D[t]) for t in TABLES})

## 1. 品質検査

数字を読む前に、解釈を壊す条件だけ確認する。ここでは4項目に絞る。

| 検出項目 | なぜ問題か |
|---|---|
| ダミー応答の混入 | API失敗時のフォールバック。プロンプト版の評価にならない |
| surveys の重複 | 回答数 n が水増しされる |
| `prompt_version` 欠損 | 版比較に使えない |
| v0 の混入 | ベースライン専用。通常の版比較に混ぜると平均が歪む |

In [ ]:
# ダミー応答の文面。ai_backend._DUMMY_REPLIES と同じものを持つ。
# ai_backend 側を変えたらここも合わせること。
DUMMY_REPLIES = {
    "うーん、考えたことなかったな。なんでそんなこと聞くの?",
    "あー、それね。まあ普通かな。",
    "え、急にどうした(笑)",
    "わからん。てきとうに答えていい?",
    "それ難しい質問だね。んー、たぶんそう思う。",
    "あんまり覚えてないけど、好きだった気がする。",
}

flags = defaultdict(set)

# 1) ダミー応答の混入
if len(messages):
    dummy_gids = set(messages.loc[
        messages["content"].isin(DUMMY_REPLIES), "game_id"])
    for g in dummy_gids:
        flags[g].add("dummy_response")
else:
    dummy_gids = set()

# 2) surveys の重複
if len(surveys):
    dup = (surveys.groupby(["game_id", "respondent_id"]).size()
                  .loc[lambda s: s > 1])
    for g, _ in dup.index:
        flags[g].add("survey_duplicate")
else:
    dup = pd.Series(dtype=int)

# 3) prompt_version 欠損
miss_pv = set(games.loc[games["prompt_version"].isna() |
                        (games["prompt_version"] == ""), "game_id"])
for g in miss_pv:
    flags[g].add("no_prompt_version")

# 4) v0 の混入
v0_gids = set(games.loc[games["prompt_version"] == "v0", "game_id"])
for g in v0_gids:
    flags[g].add("v0_baseline")

print(f"ダミー応答を含むゲーム   : {len(dummy_gids)}")
print(f"surveys が重複した組     : {len(dup)}"
      + (f" (最大 {int(dup.max())} 行/組)" if len(dup) else ""))
print(f"prompt_version 欠損      : {len(miss_pv)}")
print(f"v0(素のモデル)          : {len(v0_gids)}")
print(f"フラグ付きゲーム         : {len(flags)} / {len(games)}")

qc = pd.DataFrame([{"game_id": g, "flags": ";".join(sorted(f))}
                   for g, f in sorted(flags.items())])
qc.head(20)

In [ ]:
# ---- 除外の適用 ----
excluded = set()
if EXCLUDE_DUMMY:
    excluded |= dummy_gids
if EXCLUDE_V0:
    excluded |= v0_gids
if ENTRY_TYPE:
    excluded |= set(games.loc[games["entry_type"] != ENTRY_TYPE, "game_id"])

G = games[~games["game_id"].isin(excluded)].copy()
P = players[~players["game_id"].isin(excluded)].copy()
M = messages[~messages["game_id"].isin(excluded)].copy()
A = annotations[~annotations["game_id"].isin(excluded)].copy() if len(annotations) else annotations
# surveys は重複排除してから使う(重複バグ対策)
S = surveys[~surveys["game_id"].isin(excluded)].copy() if len(surveys) else surveys
if len(S):
    S = S.drop_duplicates(subset=["game_id", "respondent_id"], keep="first")

print(f"除外 {len(excluded)} ゲーム → 残り {len(G)} ゲーム / "
      f"発言 {len(M)} / アンケート {len(S)}")
G["prompt_version"].value_counts(dropna=False)

## 2. 判定の内訳

「真の正体 × 判定ラベル」のクロス表を作る。
正答率という1つの数字に潰す前に、**どの方向に間違えたか**を見る。

判定1件を1行にする。1対1は1ゲーム1行、グループは1ゲームn行。
`human_as_ai`(AIのふりをした人間)は `human` に畳まず別カテゴリとして残す。
畳むと「AIのふり条件の効果」が誤警報率に溶けて消えるため。

In [ ]:
gmap = G.set_index("game_id").to_dict("index")

rows = []
for _, p in P.iterrows():
    g = gmap.get(p["game_id"])
    if g is None:
        continue
    correct = p.get("correct")
    judged = p.get("judged_as")
    target = p.get("player_id")

    if pd.notna(judged):                       # グループ: 明示ラベルあり
        truth = p.get("role")
    elif pd.notna(correct) and pd.notna(p.get("vote_target")):
        target = p["vote_target"]              # 1対1: 判定先は投票先
        truth = g.get("ground_truth_identity")
        if truth is None or (isinstance(truth, float) and pd.isna(truth)):
            continue
        judged = truth if correct else ("human" if truth == "ai" else "ai")
    else:
        continue
    if pd.isna(correct):
        continue

    truth_is_ai = (truth == "ai")
    judged_ai = (judged == "ai")
    if truth_is_ai and judged_ai:        outcome = "hit"                # AIを見抜いた
    elif truth_is_ai and not judged_ai:  outcome = "miss"               # AIが人間に見えた
    elif not truth_is_ai and judged_ai:  outcome = "false_alarm"        # 人間がAIに見えた
    else:                                outcome = "correct_rejection"  # 人間を人間と判定

    rows.append({
        "game_id": p["game_id"], "target_player_id": target,
        "entry_type": g.get("entry_type"),
        "prompt_version": g.get("prompt_version"),
        "truth": truth, "judged_as": judged,
        "outcome": outcome, "correct": int(bool(correct)),
        "created_at": g.get("created_at"),
    })

J = pd.DataFrame(rows)
print(f"判定 {len(J)} 件")

cross = pd.crosstab(J["truth"], J["judged_as"],
                    margins=True, margins_name="計")
display(cross)

print("\n判定結果の内訳:")
display(J["outcome"].value_counts().rename("件数").to_frame())

In [ ]:
# 版ごとの内訳(行方向の割合)
if len(J):
    t = pd.crosstab(J["prompt_version"], J["outcome"])
    display(t)
    display((t.div(t.sum(axis=1), axis=0) * 100).round(1)
            .rename(columns=lambda c: c + "(%)"))

## 3. 誤警報の分離保存

`false_alarm` = **人間を「AI」と判定した事例**。

正答率に混ぜると消えてしまうが、研究上はここが重要になる。
「人間のどういう振る舞いがAIっぽく見えるか」は、
「AIのどういう振る舞いがAIっぽいか」とは別の問いで、
プロンプト改訂の目標(=人間らしさの基準)を決める材料になる。

ここでは誤警報の事例を切り出して CSV に保存し、会話と自由記述を読める形にする。

In [ ]:
FA = J[J["outcome"] == "false_alarm"].copy()
CR = J[J["outcome"] == "correct_rejection"].copy()
print(f"誤警報 {len(FA)} 件 / 正しく人間と判定 {len(CR)} 件 "
      f"(人間が判定対象になった {len(FA) + len(CR)} 件中)")

if len(FA):
    FA.to_csv(f"{OUT_DIR}/false_alarms.csv", index=False,
              encoding="utf-8-sig")
    print(f"→ {OUT_DIR}/false_alarms.csv に保存")

# 誤警報された人間の回答と、そうでない人間の回答を比べる
ans = M[M["kind"] == "answer"].copy()
target_outcome = J.set_index(["game_id", "target_player_id"])["outcome"]
ans["outcome"] = [
    target_outcome.get((r.game_id, r.speaker_id)) for r in ans.itertuples()]

hum = ans[ans["outcome"].isin(["false_alarm", "correct_rejection"])]
if len(hum):
    display(hum.groupby("outcome")[["char_count", "compose_time_ms"]]
               .describe().round(1).T)

In [ ]:
# 誤警報された会話を読む(先頭5件)
def show_game(gid, max_rows=None):
    g = gmap.get(gid, {})
    print(f"=== {gid}  [{g.get('entry_type')}] "
          f"版={g.get('prompt_version')} 正体={g.get('ground_truth_identity')} ===")
    rows = M[M["game_id"] == gid].sort_values(["turn", "id"])
    if max_rows:
        rows = rows.head(max_rows)
    for r in rows.itertuples():
        mark = "Q" if r.kind == "question" else "A"
        print(f"  {r.turn} {mark} [{r.speaker_id}] {r.content}")
    s = S[S["game_id"] == gid] if len(S) else pd.DataFrame()
    for r in s.itertuples():
        print(f"  --- 自由記述(確信度{r.confidence}, 根拠={r.basis}): {r.free_text}")
    print()

for gid in FA["game_id"].head(5):
    show_game(gid)

## 4. 自由記述の分析

`surveys.basis`(選択された根拠)と `surveys.free_text` / `annotations.free_text`
(自由入力)を読む。

日本語の分かち書きには形態素解析器が要るが、**入っていなくても動くようにしてある**。
`janome`(pure Python, `pip install janome`)があればそれを使い、
無ければ正規表現による簡易切り出し + 文字N-gramで代用する。
簡易版は精度が落ちるので、**頻出語は必ず原文に戻って確認すること**。

In [ ]:
# ---- 分かち書き ----
try:
    from janome.tokenizer import Tokenizer
    _t = Tokenizer()
    HAS_JANOME = True
except Exception:
    HAS_JANOME = False

# 内容語になりにくい語。頻出語リストがこれで埋まるのを防ぐ。
STOP = set(
    "する ある いる なる こと それ これ あれ どれ ここ そこ もの ため よう "
    "です ます でした ました ない なく なっ しま した して いう 思う 感じ "
    "の に は が を と も で や か ね よ な つ た だ し て れ ら せ さ "
    "あと でも けど から まで より ほど など そう どう なん てる ちょっと"
    .split())

_KANJI = re.compile(r"[一-龥ヶ]{1,8}")
_KATA  = re.compile(r"[ァ-ヴー]{2,}")
_ALPHA = re.compile(r"[A-Za-z]{2,}")
# ひらがなだけの連なりは助詞・助動詞が大半で、内容語として拾うと
# 「がうすい」「じがした」のような切れ方をした断片で表が埋まる。
# 簡易版では拾わず、口語の傾向は文字N-gram側で見る。

def tokenize(text):
    # 内容語らしきものを返す。janome があれば形態素、無ければ正規表現。
    if not isinstance(text, str) or not text.strip():
        return []
    text = unicodedata.normalize("NFKC", text)
    if HAS_JANOME:
        out = []
        for tok in _t.tokenize(text):
            pos = tok.part_of_speech.split(",")[0]
            if pos in ("名詞", "動詞", "形容詞", "副詞"):
                w = tok.base_form if tok.base_form != "*" else tok.surface
                if len(w) >= 2 and w not in STOP:
                    out.append(w)
        return out
    out = []
    for pat in (_KANJI, _KATA, _ALPHA):
        out += [w for w in pat.findall(text) if w not in STOP]
    return out

def char_ngrams(text, n=2):
    # ひらがな中心の口語は分かち書きが効きにくいので文字N-gramも併用する。
    if not isinstance(text, str):
        return []
    t = re.sub(r"\s+", "", unicodedata.normalize("NFKC", text))
    return [t[i:i+n] for i in range(len(t) - n + 1)]

def freq_table(texts, topn=25, fn=tokenize):
    c = Counter()
    for t in texts:
        c.update(fn(t))
    return pd.DataFrame(c.most_common(topn), columns=["語", "件数"])

print("形態素解析:", "janome" if HAS_JANOME else "簡易(正規表現)")
if not HAS_JANOME:
    print("  → `pip install janome` を強く推奨。"
          "簡易版は漢字・カタカナの連なりを切り出すだけなので、"
          "「何食」「最近見」のように語の途中で切れる。")

In [ ]:
# 4-1. 選択式の根拠 basis の度数
if len(S) and "basis" in S:
    b = S["basis"].dropna().replace("", pd.NA).dropna()
    display(b.value_counts().rename("件数").to_frame())
else:
    print("basis のデータがありません")

In [ ]:
# 4-2. 自由記述の頻出語(全体)
free_texts = []
if len(S) and "free_text" in S:
    free_texts += [t for t in S["free_text"].fillna("") if t.strip()]
if len(A) and "free_text" in A:
    free_texts += [t for t in A["free_text"].fillna("") if t.strip()]

print(f"自由記述 {len(free_texts)} 件")
if free_texts:
    display(freq_table(free_texts, 25))
    print("文字2-gram(分かち書きの取りこぼし確認用):")
    display(freq_table(free_texts, 20, fn=lambda t: char_ngrams(t, 2)))

In [ ]:
# 4-3. 判定結果ごとに自由記述を分けて比較する
#      「AIだと思った理由」と「人間だと思った理由」は別物なので分けて読む。
if len(S):
    s2 = S.merge(J[["game_id", "outcome", "truth", "judged_as"]],
                 on="game_id", how="left")
    for oc in ["hit", "false_alarm", "miss", "correct_rejection"]:
        sub = s2[s2["outcome"] == oc]
        texts = [t for t in sub["free_text"].fillna("") if t.strip()]
        print(f"--- {oc}  (自由記述 {len(texts)} 件) ---")
        if texts:
            display(freq_table(texts, 12))
        else:
            print("  記述なし\n")

In [ ]:
# 4-4. 特徴語 ― 「AIと判定したとき」に相対的に多い語
#      単純な頻度差ではなく対数オッズ比(加算平滑化)で見る。
#      参考: Monroe, Colaresi & Quinn (2008) Political Analysis 16(4):372-403.
import math

def log_odds(texts_a, texts_b, topn=15, min_count=2):
    ca, cb = Counter(), Counter()
    for t in texts_a: ca.update(set(tokenize(t)))   # 文書頻度で数える
    for t in texts_b: cb.update(set(tokenize(t)))
    na, nb = sum(ca.values()), sum(cb.values())
    if na == 0 or nb == 0:
        return pd.DataFrame(columns=["語", "log-odds", "A", "B"])
    vocab = {w for w in set(ca) | set(cb) if ca[w] + cb[w] >= min_count}
    rows = []
    for w in vocab:
        pa = (ca[w] + 1) / (na + len(vocab))
        pb = (cb[w] + 1) / (nb + len(vocab))
        rows.append({"語": w,
                     "log-odds": math.log(pa / (1 - pa)) - math.log(pb / (1 - pb)),
                     "A": ca[w], "B": cb[w]})
    df = pd.DataFrame(rows).sort_values("log-odds", ascending=False)
    if len(df) <= 2 * topn:      # 語彙が少ないときは頭と尾が重なるので全件返す
        return df
    return pd.concat([df.head(topn), df.tail(topn)])

if len(S):
    said_ai = [t for t in s2.loc[s2["judged_as"] == "ai", "free_text"].fillna("") if t.strip()]
    said_hu = [t for t in s2.loc[s2["judged_as"] == "human", "free_text"].fillna("") if t.strip()]
    print(f"「AI」と判定 {len(said_ai)}件 vs 「人間」と判定 {len(said_hu)}件")
    print("上=AI判定に特徴的, 下=人間判定に特徴的")
    display(log_odds(said_ai, said_hu))

## 5. 質問内容の分析

判定者が**何を聞いたか**。固定5ターン・役割固定なので、
ターン番号は版やゲームをまたいで比較できる(この設計の利点)。

In [ ]:
Q = M[M["kind"] == "question"].copy()
Q = Q.merge(J[["game_id", "outcome", "truth", "correct"]].drop_duplicates("game_id"),
            on="game_id", how="left")
print(f"質問 {len(Q)} 件 / {Q['game_id'].nunique()} ゲーム")

print("全体の頻出語:")
display(freq_table(Q["content"].fillna(""), 25))

In [ ]:
# 5-1. ターン位置ごとの頻出語 ― 「最初に何を聞き、最後に何を聞くか」
for t in sorted(Q["turn"].dropna().unique()):
    sub = Q[Q["turn"] == t]
    print(f"--- ターン {int(t)} (n={len(sub)}, 中央値 {sub['char_count'].median():.0f}字) ---")
    display(freq_table(sub["content"].fillna(""), 8))

In [ ]:
# 5-2. 規則ベースの質問タイプ分類
#      これは探索用の粗い分類。カテゴリの妥当性は必ず手作業で確認すること
#      (一致率を取るなら別の人にも同じ分類をさせてκを計算する)。
PATTERNS = [
    ("メタ(正体を直接聞く)", r"AI|えーあい|人間|ロボット|ボット|機械|本物|正体"),
    ("命令・形式指定",       r"答えて|言って|してみて|してください|で答え|以内で|文字で|繰り返"),
    ("記憶・経験",           r"昨日|今朝|さっき|最近|子供の頃|覚えて|思い出|したこと|行った|食べた"),
    ("感情・主観",           r"好き|嫌い|楽しい|つらい|うれし|嬉し|怖|むかつ|どう思|気持ち|感じる"),
    ("知識・計算",           r"何年|いくつ|計算|足す|引く|意味|とは|説明|理由"),
    ("身体・生活",           r"寝|食べ|飲|お腹|眠|疲れ|バイト|学校|仕事|住ん"),
    ("雑談・好み",           r"趣味|音楽|ゲーム|映画|animeアニメ|休み|天気"),
]

def classify(text):
    if not isinstance(text, str):
        return "その他"
    t = unicodedata.normalize("NFKC", text)
    for name, pat in PATTERNS:
        if re.search(pat, t, re.I):
            return name
    return "その他"

Q["qtype"] = Q["content"].map(classify)
display(Q["qtype"].value_counts().rename("件数").to_frame())

# ターン位置 × 質問タイプ
display(pd.crosstab(Q["turn"], Q["qtype"]))

In [ ]:
# 5-3. 質問タイプ別に、そのゲームの判定が当たったか
#      注意: 1ゲームに5問あるので「その質問のおかげで当たった」とは言えない。
#      あくまで「当たったゲームに多く現れた質問タイプ」という相関。
if Q["correct"].notna().any():
    tab = (Q.dropna(subset=["correct"])
             .groupby("qtype")["correct"]
             .agg(質問数="size", 正答ゲームの割合="mean")
             .sort_values("質問数", ascending=False))
    tab["正答ゲームの割合"] = tab["正答ゲームの割合"].round(3)
    display(tab)

In [ ]:
# 5-4. 当たったゲームと外したゲームで、質問の語がどう違うか
if Q["correct"].notna().any():
    qa = [t for t in Q.loc[Q["correct"] == 1, "content"].fillna("") if t.strip()]
    qb = [t for t in Q.loc[Q["correct"] == 0, "content"].fillna("") if t.strip()]
    print(f"正答ゲームの質問 {len(qa)}件 vs 誤答ゲームの質問 {len(qb)}件")
    print("上=正答に特徴的, 下=誤答に特徴的")
    display(log_odds(qa, qb))

## 6. 決め手アノテーション

「どの発言が決め手／違和感だったか」の注釈。
フロントエンドが未実装なら空になる。

In [ ]:
if len(A):
    display(A["kind"].value_counts().rename("件数").to_frame())
    AM = A.merge(M[["id", "content", "speaker_id", "turn", "kind"]]
                   .rename(columns={"id": "message_id", "kind": "msg_kind"}),
                 on="message_id", how="left")
    display(AM[["game_id", "turn", "msg_kind", "speaker_id",
                "kind", "content", "free_text"]].head(30))
else:
    print("注釈データがありません(アノテーションUIが未実装)")

## 7. 会話全文ビューア

`show_game("ゲームID")` で1件読む。読み込み対象を切り替えたいときはここを使う。

In [ ]:
# 誤警報のうち自由記述があるものを優先して表示する
cands = list(FA["game_id"]) if len(FA) else list(J["game_id"].head(3))
for gid in cands[:3]:
    show_game(gid)

## 8. 書き出し

以降の作業(手作業でのコーディング、R での検定など)に渡すためのファイル。

In [ ]:
J.to_csv(f"{OUT_DIR}/judgments.csv", index=False, encoding="utf-8-sig")
Q[["game_id", "turn", "content", "char_count", "compose_time_ms",
   "qtype", "outcome", "correct"]].to_csv(
    f"{OUT_DIR}/questions.csv", index=False, encoding="utf-8-sig")
if len(S):
    S.merge(J[["game_id", "outcome", "truth", "judged_as"]], on="game_id",
            how="left").to_csv(f"{OUT_DIR}/surveys.csv", index=False,
                               encoding="utf-8-sig")
if len(qc):
    qc.to_csv(f"{OUT_DIR}/qc_flags.csv", index=False, encoding="utf-8-sig")
print("書き出し:", sorted(os.listdir(OUT_DIR)))

---

## 読むときの注意

- **頻出語は原文に戻って確認する。** 形態素解析なしの簡易切り出しは
  語の途中で切れることがある。表に出た語は必ず `show_game()` で文脈を見る。
- **質問タイプ分類は正規表現の当て推量。** カテゴリを研究の結論に使うなら、
  自分で全件ラベル付けし、規則との一致率を確認すること。
- **5-3 の「正答ゲームの割合」は因果ではない。** 1ゲームに質問が5問あるため、
  どの質問が効いたかはこの表からは分からない。決め手を特定するには
  アノテーション(第6節)が要る。
- **探索で見つけた差をそのまま検定にかけない。** ここで見つけたものは
  次のデータ収集で確かめる仮説として扱う。